# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets\\scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25

In [8]:
from rank_bm25 import BM25Okapi
import numpy as np
import math

# Preparar el corpus para BM25
# Extraer los IDs para mantener la referencia y tokenizamos el texto
doc_ids = list(corpus.keys())

# Concatenamos el titulo y el texto para un mejor contexto de busqueda
tokenized_corpus = [
    (corpus[d].get("title", "") + " " + corpus[d].get("text", "")).lower().split() 
    for d in doc_ids
]

bm25 = BM25Okapi(tokenized_corpus)

# Generar el ranking inicial Top 10 para cada query
top_k = 10
retrieval_results = {}

for qid, query_text in queries.items():
    # Tokenizamos la query
    tokenized_query = query_text.lower().split()
    
    # Obtenemos los scores para todos los documentos del corpus
    scores = bm25.get_scores(tokenized_query)
    
    # Obtenemos los indices de los 'top_k' documentos con mayor score
    top_k_indices = np.argsort(scores)[::-1][:top_k]
    
    # Guardamos los resultados en un diccionario {doc_id: score}
    retrieval_results[qid] = {doc_ids[i]: scores[i] for i in top_k_indices}

print("Retrieval inicial completado con exito.")

Retrieval inicial completado con exito.


* Obtener métricas: Recall@10 DCG@10

In [10]:
# Funcion de evaluacion para Recall@K y DCG@K
def evaluate_baseline(qrels, results, k=10):
    total_recall = 0.0
    total_dcg = 0.0
    valid_queries = 0

    for qid, relevant_docs in qrels.items():
        # Filtrar los documentos que realmente son relevantes (rel > 0)
        total_rels = sum([1 for doc_id, rel in relevant_docs.items() if rel > 0])
        
        # Ignoramos queries que no tienen documentos relevantes en el ground truth
        if total_rels == 0:
            continue

        valid_queries += 1
        retrieved_docs = results.get(qid, {})

        # Ordenamos los documentos recuperados por su score BM25 (de mayor a menor)
        sorted_retrieved = sorted(retrieved_docs.items(), key=lambda x: x[1], reverse=True)[:k]

        # Calculo Recall@K
        # Contamos cuantos de los recuperados estan en el set de relevantes
        hits = sum([1 for doc_id, _ in sorted_retrieved if doc_id in relevant_docs and relevant_docs[doc_id] > 0])
        total_recall += (hits / total_rels)

        # Calculo de DCG@K
        dcg = 0.0
        for i, (doc_id, _) in enumerate(sorted_retrieved):
            rel = relevant_docs.get(doc_id, 0)
            if rel > 0:
                # log2(i + 2) para simular log2(i + 1) por comportamiento de python
                dcg += rel / math.log2(i + 2) 
        
        total_dcg += dcg

    # Retornamos los promedios
    return total_recall / valid_queries, total_dcg / valid_queries

# Ejecutamos la evaluacion
recall_10, dcg_10 = evaluate_baseline(qrels, retrieval_results, k=10)

print(f"Métricas del Baseline (BM25)")
print(f"Recall@10: {recall_10:.4f}")
print(f"DCG@10:    {dcg_10:.4f}")

Métricas del Baseline (BM25)
Recall@10: 0.6862
DCG@10:    0.5929


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.

In [11]:
from sentence_transformers import CrossEncoder
import pandas as pd

# Cargar el modelo Cross-Encoder pre-entrenado
model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
cross_encoder = CrossEncoder(model_name, max_length=512)

reranked_results = {}
position_changes = [] 

print("Iniciando el re-ranking con Cross-Encoder...")

# Re-rankear los top-k candidatos obtenidos con BM25
for qid, bm25_docs in retrieval_results.items():
    query_text = df_queries.loc[df_queries["query_id"] == qid, "query"].values[0]
    
    # Obtenemos los IDs del top-10 recuperado por BM25, manteniendo su orden original
    top_doc_ids = list(bm25_docs.keys())
    
    # Preparamos los pares [query, document] para el Cross-Encoder
    pairs = []
    for doc_id in top_doc_ids:
        # Combinamos titulo y texto del documento al igual que hicimos con BM25
        doc_data = corpus.get(doc_id, {})
        doc_text = doc_data.get("title", "") + " " + doc_data.get("text", "")
        pairs.append([query_text, doc_text])
        
    # Calculamos los scores semanticos para los 10 pares
    ce_scores = cross_encoder.predict(pairs)
    
    # Asociamos cada doc_id con su nuevo score y su posicion original (BM25)
    # start=1 para que los rangos vayan del 1 al 10 en lugar de 0 a 9
    doc_score_ranks = [
        (doc_id, float(score), bm25_rank) 
        for bm25_rank, (doc_id, score) in enumerate(zip(top_doc_ids, ce_scores), start=1)
    ]
    
    # Ordenamos de mayor a menor basandonos exclusivamente en el nuevo score del Cross-Encoder
    doc_score_ranks.sort(key=lambda x: x[1], reverse=True)
    
    # Guardamos el nuevo ranking en el diccionario final
    reranked_results[qid] = {doc_id: score for doc_id, score, _ in doc_score_ranks}
    
    # Identificar que documentos cambian de posicion
    for new_ce_rank, (doc_id, score, old_bm25_rank) in enumerate(doc_score_ranks, start=1):
        if old_bm25_rank != new_ce_rank:
            # Calculamos el desplazamiento: si es positivo, subio posiciones; si es negativo, bajo.
            shift = old_bm25_rank - new_ce_rank 
            position_changes.append({
                "query_id": qid,
                "doc_id": doc_id,
                "bm25_rank": old_bm25_rank,
                "ce_rank": new_ce_rank,
                "posiciones_escaladas": shift 
            })

print("Re-ranking completado con exito.")

Re-ranking completado con exito.


* Identificar qué documentos cambian de posición en el top 10


In [12]:
# Convertimos la lista de cambios a un DataFrame
df_changes = pd.DataFrame(position_changes)

if not df_changes.empty:
    print("Resumen de documentos que cambiaron de posicion tras el re-ranking:")
    # Filtramos para ver los documentos que mas posiciones ganaron
    df_mejorados = df_changes.sort_values(by="posiciones_escaladas", ascending=False)
    display(df_mejorados.head(15))
else:
    print("Los rankings de BM25 y Cross-Encoder fueron identicos para el Top-10")

Resumen de documentos que cambiaron de posicion tras el re-ranking:


,query_id,doc_id,bm25_rank,ce_rank,posiciones_escaladas
1508,870,4345757,10,1,9
1429,830,23634484,10,1,9
1937,1180,31272411,10,1,9
514,295,5979056,10,2,8
237,133,6969753,10,2,8
1883,1140,12009265,9,1,8
1820,1100,22420524,10,2,8
531,300,40721190,10,2,8
1553,887,9641846,9,1,8
994,577,35321950,9,1,8


Los resultados demuestran la eficacia del modelo neuronal al corregir las limitaciones léxicas de BM25, elevando a las primeras posiciones documentos semánticamente relevantes que inicialmente habían sido relegados al fondo del Top-10.

## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

1. Extracción de Features y Entrenamiento LTR

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np

# Feature Engineering para el Top-10 de BM25
ltr_features = []

print("Extrayendo características para LTR...")
for qid, bm25_docs in retrieval_results.items():
    query_text = df_queries.loc[df_queries["query_id"] == qid, "query"].values[0].lower()
    q_tokens = set(query_text.split())
    q_len = len(q_tokens)
    
    for doc_id, bm25_score in bm25_docs.items():
        doc_data = corpus.get(doc_id, {})
        doc_text = (doc_data.get("title", "") + " " + doc_data.get("text", "")).lower()
        d_tokens = set(doc_text.split())
        d_len = len(d_tokens)
        
        # Target: La relevancia real extraída de qrels (0 si no es relevante)
        rel = qrels.get(qid, {}).get(doc_id, 0)
        
        # Feature: Overlap (cuantas palabras comparten la query y el documento)
        term_overlap = len(q_tokens.intersection(d_tokens))
        
        ltr_features.append({
            "query_id": qid,
            "doc_id": doc_id,
            "bm25_score": bm25_score,       # Feature 1
            "doc_length": d_len,            # Feature 2
            "query_length": q_len,          # Feature 3
            "term_overlap": term_overlap,   # Feature 4
            "relevance": rel                # Target
        })

df_ltr = pd.DataFrame(ltr_features)

# Preparar los datos para XGBoost
# XGBRanker requiere que los datos esten ordenados por el grupo (query_id)
df_ltr = df_ltr.sort_values(by="query_id")

features = ["bm25_score", "doc_length", "query_length", "term_overlap"]
X = df_ltr[features]
y = df_ltr["relevance"]

# Calculamos cuantos documentos hay por cada query para agruparlos
groups = df_ltr.groupby("query_id").size().values

# Entrenar el modelo Ranker
# objective="rank:ndcg" le dice a XGBoost que optimice directamente la metrica de ranking
ranker = xgb.XGBRanker(
    tree_method="hist",
    objective="rank:ndcg", 
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

print("Entrenando el modelo XGBRanker...")
ranker.fit(X, y, group=groups)

2. Predicción y Análisis de Posiciones

In [14]:
# Generamos los nuevos scores LTR
df_ltr["ltr_score"] = ranker.predict(X)

ltr_reranked_results = {}
ltr_position_changes = []

print("Calculando los nuevos rankings...")
for qid, group in df_ltr.groupby("query_id"):
    # Reconstruir el orden original de BM25 para comparar
    original_order = group.sort_values(by="bm25_score", ascending=False)["doc_id"].tolist()
    
    # Ordenamos segun las predicciones del modelo LTR
    new_order = group.sort_values(by="ltr_score", ascending=False)
    
    ltr_reranked_results[qid] = {}
    
    for new_rank, (index, row) in enumerate(new_order.iterrows(), start=1):
        doc_id = row["doc_id"]
        # Guardar el resultado
        ltr_reranked_results[qid][doc_id] = row["ltr_score"]
        
        # Identifir la posicion anterior
        old_rank = original_order.index(doc_id) + 1
        
        if old_rank != new_rank:
            ltr_position_changes.append({
                "query_id": qid,
                "doc_id": doc_id,
                "bm25_rank": old_rank,
                "ltr_rank": new_rank,
                "posiciones_escaladas": old_rank - new_rank
            })

# Mostrar la tabla de posiciones que cambiaron
df_ltr_changes = pd.DataFrame(ltr_position_changes)

if not df_ltr_changes.empty:
    print("\nResumen de documentos que cambiaron de posicion tras el LTR (XGBoost):")
    df_mejorados_ltr = df_ltr_changes.sort_values(by="posiciones_escaladas", ascending=False)
    display(df_mejorados_ltr.head(15))
else:
    print("\nEl modelo LTR mantuvo el mismo orden que BM25.")

Calculando los nuevos rankings...

Resumen de documentos que cambiaron de posicion tras el LTR (XGBoost):


,query_id,doc_id,bm25_rank,ltr_rank,posiciones_escaladas
376,1226,21909315,10,1,9
572,1303,195352,10,1,9
1481,544,10648422,10,1,9
256,1180,31272411,10,1,9
1542,569,8038329,10,2,8
1038,312,6173523,9,1,8
604,1320,19204979,10,2,8
1269,478,14474178,10,2,8
1268,478,23801039,9,1,8
2105,873,41310252,9,1,8


Al igual que el modelo neuronal, el enfoque LTR (XGBoost) demuestra su capacidad para rescatar documentos relevantes que el filtro inicial había penalizado. Al evaluar un conjunto de características estadísticas más rico en lugar de depender únicamente de la frecuencia de términos, el modelo aprende a corregir las limitaciones léxicas y eleva al Top-1 y Top-2 resultados que originalmente estaban relegados al final del ranking.

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* DCG@10
* MAP
* Recall@10

In [15]:
import math

# Definir la funcion de evaluacion con las 3 metricas
def evaluate_all_metrics(qrels, results, k=10):
    total_recall = 0.0
    total_dcg = 0.0
    total_ap = 0.0
    valid_queries = 0

    for qid, relevant_docs in qrels.items():
        # Filtramos los documentos que son relevantes
        total_rels = sum([1 for doc_id, rel in relevant_docs.items() if rel > 0])
        
        if total_rels == 0:
            continue

        valid_queries += 1
        retrieved_docs = results.get(qid, {})

        # Ordenar por score de mayor a menor
        sorted_retrieved = sorted(retrieved_docs.items(), key=lambda x: x[1], reverse=True)[:k]

        hits = 0
        dcg = 0.0
        sum_precisions = 0.0

        for i, (doc_id, _) in enumerate(sorted_retrieved):
            rel = relevant_docs.get(doc_id, 0)
            if rel > 0:
                hits += 1
                
                # DCG@K
                dcg += rel / math.log2(i + 2)
                
                # Average Precision (AP)
                # Precision en el punto 'i' 
                precision_at_i = hits / (i + 1)
                sum_precisions += precision_at_i

        # Recall@K
        total_recall += (hits / total_rels)
        
        # Sumando DCG
        total_dcg += dcg
        
        # AP (dividiendo por el total de relevantes de la query)
        total_ap += (sum_precisions / total_rels)

    # Retornamos los promedios sobre todas las queries validas
    return {
        f"Recall@{k}": total_recall / valid_queries,
        f"DCG@{k}": total_dcg / valid_queries,
        "MAP": total_ap / valid_queries
    }

# Calcular metricas para los 3 enfoques
print("Evaluando modelos...")
metrics_bm25 = evaluate_all_metrics(qrels, retrieval_results, k=10)
metrics_ce   = evaluate_all_metrics(qrels, reranked_results, k=10)
metrics_ltr  = evaluate_all_metrics(qrels, ltr_reranked_results, k=10)

# Construir un DataFrame para ver la comparativa clara
df_metrics = pd.DataFrame([
    {"Modelo": "1. Baseline (BM25)", **metrics_bm25},
    {"Modelo": "2. Re-ranking: Cross-Encoder", **metrics_ce},
    {"Modelo": "3. Re-ranking: LTR (XGBoost)", **metrics_ltr}
])

# Mostramos los resultados redondeados para mejor lectura
df_metrics.round(4)

Evaluando modelos...


,Modelo,Recall@10,DCG@10,MAP
0,1. Baseline (BM25),0.6862,0.5929,0.5147
1,2. Re-ranking: Cross-Encoder,0.6862,0.6577,0.5882
2,3. Re-ranking: LTR (XGBoost),0.6862,0.7344,0.6819


La arquitectura de recuperación en dos etapas demuestra que el paso inicial de Retrieval (BM25) es excelente para filtrar millones de documentos rápidamente, definiendo el "techo" de nuestro Recall. No obstante, sus métricas de precisión sufren debido a su dependencia puramente léxica.

Al aplicar el Re-ranking sobre los candidatos principales, tanto el enfoque semántico profundo (Cross-Encoder) como el enfoque estadístico no lineal (LTR basado en árboles) logran empujar los documentos más relevantes hacia las primeras posiciones. Esto se evidencia claramente en el incremento del MAP y el DCG@10, demostrando que presentarle el documento correcto al usuario en la posición #1 o #2 genera un impacto enorme en la calidad percibida del sistema de búsqueda.